# Evaluasi
Analisis dilakukan pada data validation menggunakan prediksi yang disimpan oleh `02_training.ipynb`. Notebook ini dapat dijalankan tanpa GPU. Grad-CAM membutuhkan file checkpoint di folder `checkpoint/`.

## 0. Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Violita-archive/document-image-classification.git"
REPO_NAME = "document-image-classification"

if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working") / REPO_NAME
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    try:
        import timm
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Root:", ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from PIL import Image

from src.dataset import resolve_image_path
from src.evaluate import evaluation_report, plot_confusion_matrix, top_confusions
from src.gradcam import GradCAM, get_target_layer
from src.model import load_checkpoint
from src.transforms import get_eval_transforms, to_hw
from src.utils import CLASS_NAMES, load_config, resolve_data_dir

FIG_DIR, METRIC_DIR = Path("results/figures"), Path("results/metrics")
DATA_DIR = resolve_data_dir(load_config("configs/config.yaml")["data_dir"])
FINAL, BASE, CAM_MODEL = "ensemble", "baseline_resnet18", "convnext_tiny_512"

pred = pd.read_csv(METRIC_DIR / f"{FINAL}_val_predictions.csv")
pred_base = pd.read_csv(METRIC_DIR / f"{BASE}_val_predictions.csv")
pred["path"] = [resolve_image_path(DATA_DIR, f) for f in pred["filename"]]
print(f"Jumlah data validation: {len(pred):,}")

## 1. Ringkasan eksperimen

In [ ]:
experiments = pd.read_csv(METRIC_DIR / "experiments.csv")
display(experiments)

## 2. Metrik per kelas

In [ ]:
acc, report = evaluation_report(pred["label"], pred["pred"])
acc_b, report_b = evaluation_report(pred_base["label"], pred_base["pred"])
print(f"Akurasi validation  final: {acc:.4f} | baseline: {acc_b:.4f}")

per_class = report.loc[CLASS_NAMES, ["precision", "recall", "f1-score", "support"]].copy()
per_class["f1_baseline"] = report_b.loc[CLASS_NAMES, "f1-score"]
per_class = per_class.sort_values("f1-score")
per_class.to_csv(METRIC_DIR / "classification_report.csv")
display(per_class.style.format("{:.4f}", subset=["precision", "recall", "f1-score", "f1_baseline"])
        .background_gradient(cmap="RdYlGn", subset=["f1-score"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(per_class))
ax.barh(y - 0.2, per_class["f1_baseline"], 0.4, label="baseline ResNet-18", color="#BBBBBB")
ax.barh(y + 0.2, per_class["f1-score"], 0.4, label="ensemble ConvNeXt", color="#4C72B0")
ax.set_yticks(y, per_class.index)
ax.set(xlabel="F1-score", title="F1-score per kelas", xlim=(per_class[["f1-score", "f1_baseline"]].min().min() - 0.05, 1))
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "f1_per_class.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Confusion matrix

In [ ]:
cm = plot_confusion_matrix(pred["label"], pred["pred"], normalize=True, save_path=FIG_DIR / "confusion_matrix.png")

In [ ]:
conf_pairs = top_confusions(pred["label"], pred["pred"], k=10)
display(conf_pairs)

## 4. Tingkat keyakinan prediksi

In [ ]:
pred["benar"] = pred["label"] == pred["pred"]
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(data=pred, x="confidence", hue="benar", bins=40, stat="density", common_norm=False, ax=ax)
ax.set(title="Distribusi probabilitas prediksi: benar vs salah", xlabel="probabilitas kelas prediksi")
plt.savefig(FIG_DIR / "confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

bins = pd.cut(pred["confidence"], [0, 0.5, 0.7, 0.9, 1.0])
display(pred.groupby(bins, observed=True)["benar"].agg(["count", "mean"]).rename(columns={"count": "jumlah", "mean": "akurasi"}))

## 5. Contoh prediksi salah
Diambil kesalahan dengan probabilitas tertinggi, yaitu kasus ketika model yakin namun keliru.

In [ ]:
wrong = pred[~pred["benar"]].sort_values("confidence", ascending=False).head(16)
fig, axes = plt.subplots(2, 8, figsize=(22, 8))
for ax, (_, r) in zip(axes.ravel(), wrong.iterrows()):
    with Image.open(r["path"]) as im:
        ax.imshow(im.convert("L"), cmap="gray")
    ax.set_title(f"asli: {CLASS_NAMES[r['label']]}\nprediksi: {CLASS_NAMES[r['pred']]} ({r['confidence']:.2f})", fontsize=8)
    ax.axis("off")
plt.suptitle("Prediksi salah dengan keyakinan tertinggi")
plt.tight_layout()
plt.savefig(FIG_DIR / "misclassified_examples.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Grad-CAM
Area merah menunjukkan bagian gambar yang paling berpengaruh terhadap prediksi.

In [ ]:
ckpt_path = Path("checkpoint") / f"{CAM_MODEL}_best.pth"
if ckpt_path.exists():
    model, cfg = load_checkpoint(ckpt_path, "cpu")
    cam = GradCAM(model, get_target_layer(model))
    tfm = get_eval_transforms(cfg["img_size"])
    sample = pred[pred["benar"]].groupby("label").head(1).sort_values("label")
    sample = pd.concat([sample, wrong.head(4)])

    fig, axes = plt.subplots(4, 5, figsize=(18, 17))
    for ax, (_, r) in zip(axes.ravel(), sample.iterrows()):
        with Image.open(r["path"]) as im:
            img = im.convert("RGB")
        heat, cls, p = cam(tfm(img).unsqueeze(0))
        h, w = to_hw(cfg["img_size"])
        ax.imshow(img.resize((w, h)))
        ax.imshow(heat, cmap="jet", alpha=0.4)
        tag = "" if r["benar"] else " [salah]"
        ax.set_title(f"asli: {CLASS_NAMES[r['label']]}\nprediksi: {CLASS_NAMES[cls]} ({p:.2f}){tag}", fontsize=9)
        ax.axis("off")
    for ax in axes.ravel()[len(sample):]:
        ax.axis("off")
    plt.suptitle("Grad-CAM ConvNeXt-Tiny (512 x 384)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "gradcam_examples.png", dpi=120, bbox_inches="tight")
    plt.show()
    cam.remove()
else:
    print(f"Checkpoint {ckpt_path} tidak ditemukan, Grad-CAM dilewati.")

## 7. Skor Kaggle

| Submission | Val accuracy | Public | Private |
|---|---|---|---|
| baseline_resnet18 | | | |
| convnext_tiny_384 | | | |
| convnext_tiny_512 | | | |
| ensemble | | | |

## 8. Kesimpulan

*(diisi setelah hasil tersedia)*